In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('. Group Project/Post mid-term/44_Feature_Dataset.csv')

# 1. Basic dataset information
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# 2. Column names and data types
print("\n" + "=" * 60)
print("ALL COLUMNS AND DATA TYPES")
print("=" * 60)

for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{i:2}. {col} ({dtype})")

# 3. First and last rows
print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
print(df.head().to_string())

print("\n" + "=" * 60)
print("LAST 5 ROWS")
print("=" * 60)
print(df.tail().to_string())

# 4. Identify potential split columns
print("\n" + "=" * 60)
print("POTENTIAL TRAIN/TEST SPLIT COLUMNS")
print("=" * 60)

split_keywords = [
    "split", "train", "test", "set",
    "partition", "subset", "fold"
]

split_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in split_keywords)
]

if split_cols:
    for col in split_cols:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False))
else:
    print("No obvious train/test split column found.")

# 5. Identify possible target columns
print("\n" + "=" * 60)
print("POTENTIAL TARGET COLUMNS")
print("=" * 60)

target_keywords = [
    "icu", "transfer", "target", "label",
    "outcome", "class", "y"
]

target_cols = [
    col for col in df.columns
    if any(
        col.lower() == keyword or
        keyword in col.lower().split("_")
        for keyword in target_keywords
    )
]

for col in target_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False).head(20))

# 6. Low-cardinality columns
print("\n" + "=" * 60)
print("COLUMNS WITH <= 5 UNIQUE VALUES")
print("=" * 60)

for col in df.columns:
    if df[col].nunique(dropna=True) <= 5:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False).to_string())

# 7. Missing values
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# 8. Duplicate rows
print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

print(f"Exact duplicate rows: {df.duplicated().sum()}")

# 9. Dataset summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Potential split columns: {split_cols}")
print(f"Potential target columns: {target_cols}")

In [ ]:
import pandas as pd

df = pd.read_csv("44_Feature_Dataset.csv")

splits = {
    name: set(group["subject_id"])
    for name, group in df.groupby("dataset_split")
}

print("Train/validation overlap:",
      len(splits["train"] & splits["validation"]))

print("Train/test overlap:",
      len(splits["train"] & splits["test"]))

print("Validation/test overlap:",
      len(splits["validation"] & splits["test"]))

print("\nClass distribution by split:")
print(
    pd.crosstab(
        df["dataset_split"],
        df["icu_transfer_within_24h"],
        normalize="index"
    ).round(4)
)

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
OUTPUT_DIR = Path("no_smote_results")
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    "44_features": "44_Feature_Dataset.csv",
    "original_train": "kidney_final_train_dataset.csv",
    "original_test": "kidney_final_test_dataset.csv",
}

# The 44-feature dataset has an explicitly confirmed target.
TARGET_44 = "icu_transfer_within_24h"

# The original kidney dataset previously used this target.
TARGET_ORIGINAL = "icu_transfer"

EXCLUDE_COLUMNS = [
    "dataset_split",
    "row_id",
    "subject_id",
    "hadm_id",
    "index_time"
]

print("Output directory:", OUTPUT_DIR.resolve())

In [ ]:
def get_xy(df, target):
    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' not found. "
            f"Available columns: {df.columns.tolist()}"
        )

    excluded = set(EXCLUDE_COLUMNS + [target])
    X = df.drop(columns=list(excluded & set(df.columns)))
    y = df[target].astype(int)

    if not set(y.unique()).issubset({0, 1}):
        raise ValueError("Target must contain binary 0/1 labels.")

    return X, y


def load_44_features():
    df = pd.read_csv(FILES["44_features"])

    train = df[df["dataset_split"] == "train"].copy()
    val = df[df["dataset_split"] == "validation"].copy()
    test = df[df["dataset_split"] == "test"].copy()

    # Confirm patient-disjoint partitions.
    if "subject_id" in df.columns:
        groups = [
            set(part["subject_id"])
            for part in [train, val, test]
        ]

        assert not (groups[0] & groups[1])
        assert not (groups[0] & groups[2])
        assert not (groups[1] & groups[2])

    return (
        *get_xy(train, TARGET_44),
        *get_xy(val, TARGET_44),
        *get_xy(test, TARGET_44)
    )


def load_original():
    train_df = pd.read_csv(FILES["original_train"])
    test_df = pd.read_csv(FILES["original_test"])

    # Detect the target if its name differs.
    target = TARGET_ORIGINAL

    if target not in train_df.columns:
        candidates = [
            "icu_transfer_within_24h",
            "icu_transfer",
            "target",
            "label"
        ]
        matches = [
            c for c in candidates
            if c in train_df.columns and c in test_df.columns
        ]

        if len(matches) != 1:
            raise ValueError(
                "Cannot uniquely identify the original dataset target. "
                f"Train columns: {train_df.columns.tolist()}"
            )
        target = matches[0]

    # Split original training data into train/validation.
    if "subject_id" in train_df.columns:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=RANDOM_STATE
        )

        train_idx, val_idx = next(
            splitter.split(
                train_df,
                groups=train_df["subject_id"]
            )
        )

        fit_df = train_df.iloc[train_idx]
        val_df = train_df.iloc[val_idx]

        if "subject_id" in test_df.columns:
            assert not (
                set(train_df["subject_id"])
                & set(test_df["subject_id"])
            ), "Original train/test files share patient IDs."

    else:
        fit_df, val_df = train_test_split(
            train_df,
            test_size=0.20,
            stratify=train_df[target],
            random_state=RANDOM_STATE
        )
        print(
            "WARNING: Original dataset has no subject_id. "
            "Patient-level separation cannot be verified."
        )

    return (
        *get_xy(fit_df, target),
        *get_xy(val_df, target),
        *get_xy(test_df, target)
    )


datasets = {
    "44_features": load_44_features(),
    "original": load_original()
}

for name, data in datasets.items():
    X_train, y_train, X_val, y_val, X_test, y_test = data

    print(f"\n{name}")
    print("Training:", X_train.shape)
    print("Validation:", X_val.shape)
    print("Testing:", X_test.shape)
    print("Training positive rate:", round(y_train.mean(), 4))
    print("Validation positive rate:", round(y_val.mean(), 4))
    print("Testing positive rate:", round(y_test.mean(), 4))

In [ ]:
def build_preprocessor(X):
    numeric_cols = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_cols = [
        c for c in X.columns if c not in numeric_cols
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    transformers = [
        ("numeric", numeric_pipeline, numeric_cols)
    ]

    if categorical_cols:
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore")
            )
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_cols)
        )

    return ColumnTransformer(transformers)


def build_model(model_name, X):
    preprocessing = build_preprocessor(X)

    if model_name == "SVM":
        classifier = LinearSVC(
            C=1.0,
            class_weight=None,
            max_iter=5000,
            random_state=RANDOM_STATE,
            dual=False
        )

    elif model_name == "MLP":
        classifier = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            batch_size=512,
            learning_rate_init=0.001,
            max_iter=150,
            early_stopping=False,
            random_state=RANDOM_STATE
        )

    else:
        raise ValueError("Unknown model")

    return Pipeline([
        ("preprocess", preprocessing),
        ("classifier", classifier)
    ])

In [ ]:
def get_scores(model, X, model_name):
    if model_name == "SVM":
        return model.decision_function(X)

    return model.predict_proba(X)[:, 1]


def select_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(
        y_true, scores
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    best_idx = np.argmax(f1_values)

    return float(thresholds[best_idx])

In [ ]:
def evaluate_model(
    model,
    model_name,
    dataset_name,
    X_test,
    y_test,
    threshold
):
    scores = get_scores(model, X_test, model_name)
    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0

    results = {
        "Dataset": dataset_name,
        "Model": model_name,
        "SMOTE": "None",
        "Threshold": threshold,
        "AUROC": roc_auc_score(y_test, scores),
        "AUPRC": average_precision_score(y_test, scores),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "Specificity": specificity,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Predicted Positive": int(predictions.sum()),
        "Test Samples": len(y_test)
    }

    # Save confusion matrix.
    cm = np.array([[tn, fp], [fn, tp]])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No ICU transfer", "ICU transfer"],
        yticklabels=["No ICU transfer", "ICU transfer"]
    )

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(f"{dataset_name} — {model_name} (No SMOTE)")
    plt.tight_layout()

    prefix = f"{dataset_name}_{model_name}_no_smote"

    plt.savefig(
        OUTPUT_DIR / f"{prefix}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # Save predictions and decision scores.
    prediction_df = pd.DataFrame({
        "true_label": np.asarray(y_test),
        "score": scores,
        "predicted_label": predictions
    })

    prediction_df.to_csv(
        OUTPUT_DIR / f"{prefix}_predictions.csv",
        index=False
    )

    return results

In [ ]:
all_results = []

for dataset_name, data in datasets.items():

    X_train, y_train, X_val, y_val, X_test, y_test = data

    for model_name in ["SVM", "MLP"]:

        print("\n" + "=" * 65)
        print(f"DATASET: {dataset_name}")
        print(f"MODEL: {model_name}")
        print("SMOTE: NONE")
        print("=" * 65)

        start_time = time.time()

        model = build_model(model_name, X_train)

        print("Training model...")
        model.fit(X_train, y_train)

        print("Selecting validation threshold...")
        val_scores = get_scores(model, X_val, model_name)

        threshold = select_f1_threshold(
            y_val,
            val_scores
        )

        print(f"Selected threshold: {threshold:.6f}")

        print("Evaluating on held-out test set...")

        results = evaluate_model(
            model=model,
            model_name=model_name,
            dataset_name=dataset_name,
            X_test=X_test,
            y_test=y_test,
            threshold=threshold
        )

        elapsed = time.time() - start_time
        results["Training and Evaluation Time (s)"] = elapsed

        all_results.append(results)

        # Save fitted pipeline.
        joblib.dump(
            model,
            OUTPUT_DIR /
            f"{dataset_name}_{model_name}_no_smote.joblib"
        )

        # Save intermediate results after each experiment.
        pd.DataFrame(all_results).to_csv(
            OUTPUT_DIR / "all_results_no_smote.csv",
            index=False
        )

        print("\nRESULTS")
        for key, value in results.items():
            if isinstance(value, (float, np.floating)):
                print(f"{key}: {value:.4f}")
            else:
                print(f"{key}: {value}")

print("\nAll experiments completed.")

In [ ]:
results_df = pd.DataFrame(all_results)

display_columns = [
    "Dataset",
    "Model",
    "SMOTE",
    "Threshold",
    "AUROC",
    "AUPRC",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy",
    "Balanced Accuracy",
    "F1",
    "TN",
    "FP",
    "FN",
    "TP",
    "Predicted Positive"
]

final_table = results_df[display_columns]

display(final_table.round(4))

# Save Excel workbook.
with pd.ExcelWriter(
    OUTPUT_DIR / "SVM_MLP_No_SMOTE_Results.xlsx",
    engine="openpyxl"
) as writer:

    final_table.to_excel(
        writer,
        sheet_name="All Results",
        index=False
    )

    for dataset_name in final_table["Dataset"].unique():
        subset = final_table[
            final_table["Dataset"] == dataset_name
        ]

        subset.to_excel(
            writer,
            sheet_name=dataset_name[:31],
            index=False
        )

print("Results saved to:", OUTPUT_DIR.resolve())

In [ ]:
%pip install -q imbalanced-learn 

In [ ]:
from pathlib import Path
import time
import gc
import json
import traceback
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42

# Use None for the no-SMOTE baseline.
SMOTE_RATIOS = [None, 0.10, 0.25, 0.50, 0.75, 1.00]
MODEL_NAMES = ["SVM", "MLP"]

# Update these paths only if your filenames differ.
DATA_DIR = Path.cwd()

FILES = {
    "44_features": DATA_DIR / "44_Feature_Dataset.csv",
    "original_train": DATA_DIR / "kidney_final_train_dataset.csv",
    "original_test": DATA_DIR / "kidney_final_test_dataset.csv",
}

TARGET_44 = "icu_transfer_within_24h"
TARGET_ORIGINAL = "icu_transfer"

EXCLUDE_COLUMNS = [
    "dataset_split",
    "row_id",
    "subject_id",
    "hadm_id",
    "index_time"
]

OUTPUT_DIR = DATA_DIR / "SVM_MLP_SMOTE_Results_Corrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_MODELS = True
SAVE_PREDICTIONS = True

# Set to ["original"] for an initial smaller run.
DATASETS_TO_RUN = ["original", "44_features"]

# Set to True to run a quick pipeline test before full training.
RUN_SMOKE_TEST = True

print("Working directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

for name, path in FILES.items():
    print(f"{name}: {path.exists()} — {path.name}")

missing_files = [str(p) for p in FILES.values() if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "The following dataset files are missin."
        + "\n".join(missing_files)
    )

In [ ]:
def get_xy(df, target):

    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' not found.\n"
            f"Available columns: {df.columns.tolist()}"
        )

    y = df[target].astype(int)

    if not set(y.unique()).issubset({0, 1}):
        raise ValueError("Target must contain only 0 and 1.")

    excluded = set(EXCLUDE_COLUMNS + [target])

    X = df.drop(
        columns=[c for c in df.columns if c in excluded]
    )

    return X, y


def load_44_features():

    df = pd.read_csv(FILES["44_features"])

    required = {"train", "validation", "test"}
    actual = set(df["dataset_split"].unique())

    if actual != required:
        raise ValueError(
            f"Unexpected split labels: {actual}"
        )

    train_df = df[df["dataset_split"] == "train"].copy()
    val_df = df[df["dataset_split"] == "validation"].copy()
    test_df = df[df["dataset_split"] == "test"].copy()

    # Patient overlap verification
    if "subject_id" in df.columns:

        train_ids = set(train_df["subject_id"])
        val_ids = set(val_df["subject_id"])
        test_ids = set(test_df["subject_id"])

        assert train_ids.isdisjoint(val_ids)
        assert train_ids.isdisjoint(test_ids)
        assert val_ids.isdisjoint(test_ids)

        print("44-feature patient-disjoint split verified.")

    return (
        *get_xy(train_df, TARGET_44),
        *get_xy(val_df, TARGET_44),
        *get_xy(test_df, TARGET_44)
    )


def load_original():

    train_df = pd.read_csv(FILES["original_train"])
    test_df = pd.read_csv(FILES["original_test"])

    target = TARGET_ORIGINAL

    if target not in train_df.columns:

        candidates = [
            "icu_transfer_within_24h",
            "icu_transfer",
            "target",
            "label"
        ]

        matches = [
            c for c in candidates
            if c in train_df.columns and c in test_df.columns
        ]

        if len(matches) != 1:
            raise ValueError(
                "Could not uniquely identify target column. "
                f"Columns: {train_df.columns.tolist()}"
            )

        target = matches[0]

    if "subject_id" in train_df.columns:

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=RANDOM_STATE
        )

        train_idx, val_idx = next(
            splitter.split(
                train_df,
                train_df[target],
                groups=train_df["subject_id"]
            )
        )

        fit_df = train_df.iloc[train_idx].copy()
        val_df = train_df.iloc[val_idx].copy()

        if "subject_id" in test_df.columns:

            train_patients = set(train_df["subject_id"])
            test_patients = set(test_df["subject_id"])

            assert train_patients.isdisjoint(test_patients), (
                "Original train/test patient overlap detected."
            )

    else:

        fit_df, val_df = train_test_split(
            train_df,
            test_size=0.20,
            stratify=train_df[target],
            random_state=RANDOM_STATE
        )

        print(
            "WARNING: Original dataset has no subject_id. "
            "Patient-level separation cannot be verified."
        )

    return (
        *get_xy(fit_df, target),
        *get_xy(val_df, target),
        *get_xy(test_df, target)
    )


datasets = {
    "original": load_original(),
    "44_features": load_44_features()
}

print("\nDATASET SUMMARY")

for name, data in datasets.items():

    X_train, y_train, X_val, y_val, X_test, y_test = data

    if list(X_train.columns) != list(X_val.columns):
        raise ValueError(f"{name}: train/validation columns differ.")

    if list(X_train.columns) != list(X_test.columns):
        raise ValueError(f"{name}: train/test columns differ.")

    print("\n" + "=" * 60)
    print("Dataset:", name)
    print("Features:", X_train.shape[1])
    print("Training:", X_train.shape)
    print("Validation:", X_val.shape)
    print("Testing:", X_test.shape)

    print("Training positive rate:", round(y_train.mean(), 4))
    print("Validation positive rate:", round(y_val.mean(), 4))
    print("Testing positive rate:", round(y_test.mean(), 4))

In [ ]:
def build_model(model_name, X, smote_ratio):

    non_numeric = X.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()

    if non_numeric:
        raise ValueError(
            f"Non-numeric predictors found: {non_numeric}. "
            "These must be encoded before standard SMOTE."
        )

    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]

    if smote_ratio is not None:

        steps.append(
            (
                "smote",
                SMOTE(
                    sampling_strategy=smote_ratio,
                    random_state=RANDOM_STATE,
                    k_neighbors=5
                )
            )
        )

    if model_name == "SVM":

        classifier = LinearSVC(
            C=1.0,
            class_weight=None,
            dual=False,
            max_iter=5000,
            random_state=RANDOM_STATE
        )

    elif model_name == "MLP":

        classifier = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            batch_size=512,
            learning_rate_init=0.001,
            max_iter=150,
            early_stopping=False,
            random_state=RANDOM_STATE
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    steps.append(("classifier", classifier))

    return ImbPipeline(steps)

In [ ]:
def get_scores(model, X, model_name):

    if model_name == "SVM":
        return model.decision_function(X)

    return model.predict_proba(X)[:, 1]


def select_best_threshold(y_true, scores):

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        scores
    )

    if len(thresholds) == 0:
        raise ValueError("No valid thresholds available.")

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    best_idx = np.argmax(f1_values)

    return float(thresholds[best_idx])

In [ ]:
def evaluate_model(
    model,
    model_name,
    dataset_name,
    smote_ratio,
    X_test,
    y_test,
    threshold
):

    scores = get_scores(model, X_test, model_name)
    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0 else 0
    )

    ratio_label = (
        "None" if smote_ratio is None
        else f"{smote_ratio:.2f}"
    )

    results = {
        "Dataset": dataset_name,
        "Model": model_name,
        "SMOTE": ratio_label,
        "Threshold": threshold,
        "AUROC": roc_auc_score(y_test, scores),
        "AUPRC": average_precision_score(y_test, scores),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "Specificity": specificity,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Predicted Positive": int(predictions.sum()),
        "Test Samples": int(len(y_test))
    }

    prefix = (
        f"{dataset_name}_{model_name}_SMOTE_{ratio_label}"
    )

    # Save confusion matrix
    cm = np.array([
        [tn, fp],
        [fn, tp]
    ])

    plt.figure(figsize=(7, 5.5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No ICU Transfer", "ICU Transfer"],
        yticklabels=["No ICU Transfer", "ICU Transfer"]
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.title(
        f"{dataset_name} | {model_name} | SMOTE {ratio_label}"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"{prefix}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    # Save confusion matrix values
    pd.DataFrame(
        cm,
        index=["Actual Negative", "Actual Positive"],
        columns=["Predicted Negative", "Predicted Positive"]
    ).to_csv(
        OUTPUT_DIR / f"{prefix}_confusion_matrix.csv"
    )

    # Save test predictions
    if SAVE_PREDICTIONS:

        pd.DataFrame({
            "True Label": np.asarray(y_test),
            "Score": scores,
            "Predicted Label": predictions
        }).to_csv(
            OUTPUT_DIR / f"{prefix}_predictions.csv",
            index=False
        )

    return results

In [ ]:
if RUN_SMOKE_TEST:

    X_train, y_train, X_val, y_val, X_test, y_test = (
        datasets["44_features"]
    )

    X_small, _, y_small, _ = train_test_split(
        X_train,
        y_train,
        train_size=10000,
        stratify=y_train,
        random_state=RANDOM_STATE
    )

    test_model = build_model(
        model_name="SVM",
        X=X_small,
        smote_ratio=0.25
    )

    print("Testing corrected SVM + SMOTE pipeline...")

    test_model.fit(X_small, y_small)

    test_scores = test_model.decision_function(
        X_val.iloc[:1000]
    )

    print("Pipeline test successful.")
    print("Pipeline steps:", list(test_model.named_steps))
    print("Number of test scores:", len(test_scores))

    del test_model
    gc.collect()

In [ ]:
all_results = []
failed_experiments = []

results_path = OUTPUT_DIR / "all_smote_results.csv"
failures_path = OUTPUT_DIR / "failed_experiments.txt"

# Start a fresh benchmark in the new output directory.
if failures_path.exists():
    failures_path.unlink()

for dataset_name in DATASETS_TO_RUN:

    X_train, y_train, X_val, y_val, X_test, y_test = (
        datasets[dataset_name]
    )

    n_positive = int((y_train == 1).sum())
    n_negative = int((y_train == 0).sum())

    original_ratio = n_positive / n_negative

    print("\n" + "#" * 75)
    print("DATASET:", dataset_name)
    print("Training samples:", len(y_train))
    print("Positive samples:", n_positive)
    print("Negative samples:", n_negative)
    print("Original minority/majority ratio:", original_ratio)
    print("#" * 75)

    for model_name in MODEL_NAMES:

        for smote_ratio in SMOTE_RATIOS:

            ratio_label = (
                "None" if smote_ratio is None
                else f"{smote_ratio:.2f}"
            )

            print("\n" + "=" * 70)
            print("Dataset:", dataset_name)
            print("Model:", model_name)
            print("SMOTE:", ratio_label)
            print("=" * 70)

            # SMOTE cannot reduce the minority/majority ratio.
            if (
                smote_ratio is not None
                and smote_ratio <= original_ratio
            ):
                print(
                    "SKIPPED: Requested ratio is not above "
                    "the original minority/majority ratio."
                )
                continue

            start_time = time.time()
            model = None

            try:

                model = build_model(
                    model_name,
                    X_train,
                    smote_ratio
                )

                print("Training started...", flush=True)

                model.fit(X_train, y_train)

                print("Training completed.", flush=True)

                # Threshold selection using validation data
                val_scores = get_scores(
                    model,
                    X_val,
                    model_name
                )

                threshold = select_best_threshold(
                    y_val,
                    val_scores
                )

                print(
                    f"Selected threshold: {threshold:.6f}"
                )

                # Final test evaluation
                results = evaluate_model(
                    model=model,
                    model_name=model_name,
                    dataset_name=dataset_name,
                    smote_ratio=smote_ratio,
                    X_test=X_test,
                    y_test=y_test,
                    threshold=threshold
                )

                elapsed = time.time() - start_time

                results[
                    "Training and Evaluation Time (s)"
                ] = elapsed

                all_results.append(results)

                # Save fitted model
                if SAVE_MODELS:

                    model_path = OUTPUT_DIR / (
                        f"{dataset_name}_{model_name}_"
                        f"SMOTE_{ratio_label}.joblib"
                    )

                    joblib.dump(model, model_path)

                # Save completed results immediately
                pd.DataFrame(all_results).to_csv(
                    results_path,
                    index=False
                )

                print("\nRESULTS")

                for key, value in results.items():

                    if isinstance(
                        value,
                        (float, np.floating)
                    ):
                        print(f"{key}: {value:.4f}")
                    else:
                        print(f"{key}: {value}")

            except Exception:

                error_text = traceback.format_exc()

                failure_name = (
                    f"{dataset_name} | {model_name} | "
                    f"SMOTE {ratio_label}"
                )

                failed_experiments.append(failure_name)

                print("\nEXPERIMENT FAILED:")
                print(failure_name)
                print(error_text)

                with open(failures_path, "a") as f:
                    f.write(
                        "\n" + "=" * 70 + "\n"
                        + failure_name + "\n"
                        + error_text + "\n"
                    )

            finally:

                if model is not None:
                    del model

                gc.collect()

print("\n" + "=" * 75)
print("BENCHMARK FINISHED")
print("=" * 75)

print("Successful experiments:", len(all_results))
print("Failed experiments:", len(failed_experiments))
print("Expected experiments:", (
    len(DATASETS_TO_RUN)
    * len(MODEL_NAMES)
    * len(SMOTE_RATIOS)
))

if failed_experiments:
    print("\nFailed configurations:")
    for item in failed_experiments:
        print("-", item)

In [ ]:
if not results_path.exists():
    raise FileNotFoundError(
        "No successful experiment results were saved. "
        "Check failed_experiments.txt."
    )

results_df = pd.read_csv(
    results_path,
    dtype={"SMOTE": "string"}
)

def normalize_smote_label(value):
    if pd.isna(value) or str(value).lower() == "none":
        return "None"
    return f"{float(value):.2f}"

results_df["SMOTE"] = (
    results_df["SMOTE"].apply(normalize_smote_label)
)

metric_columns = [
    "Dataset",
    "Model",
    "SMOTE",
    "Threshold",
    "AUROC",
    "AUPRC",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy",
    "Balanced Accuracy",
    "F1",
    "TN",
    "FP",
    "FN",
    "TP",
    "Predicted Positive",
    "Test Samples",
    "Training and Evaluation Time (s)"
]

results_df = results_df[metric_columns]

display(results_df.round(4))

excel_path = OUTPUT_DIR / "SVM_MLP_SMOTE_Comparison.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="All Results",
        index=False
    )

    for dataset_name in results_df["Dataset"].unique():

        for model_name in MODEL_NAMES:

            subset = results_df[
                (results_df["Dataset"] == dataset_name)
                & (results_df["Model"] == model_name)
            ]

            if subset.empty:
                continue

            sheet_name = (
                f"{dataset_name}_{model_name}"
            )[:31]

            subset.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

print("Saved:", excel_path)

In [ ]:
ratio_order = [
    "None",
    "0.10",
    "0.25",
    "0.50",
    "0.75",
    "1.00"
]

metrics_to_compare = [
    "Threshold",
    "AUROC",
    "AUPRC",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy",
    "Balanced Accuracy",
    "F1",
    "TN",
    "FP",
    "FN",
    "TP",
    "Predicted Positive",
    "Test Samples",
    "Training and Evaluation Time (s)"
]

comparison_path = (
    OUTPUT_DIR / "SMOTE_Metric_Comparison_Tables.xlsx"
)

with pd.ExcelWriter(
    comparison_path,
    engine="openpyxl"
) as writer:

    for dataset_name in results_df["Dataset"].unique():

        for model_name in MODEL_NAMES:

            subset = results_df[
                (results_df["Dataset"] == dataset_name)
                & (results_df["Model"] == model_name)
            ]

            if subset.empty:
                continue

            table = (
                subset
                .set_index("SMOTE")[metrics_to_compare]
                .T
                .reindex(columns=ratio_order)
            )

            print("\n" + "=" * 70)
            print(dataset_name, "|", model_name)
            print("=" * 70)

            display(table.round(4))

            sheet_name = (
                f"{dataset_name}_{model_name}"
            )[:31]

            table.to_excel(
                writer,
                sheet_name=sheet_name
            )

print("\nComparison workbook saved:", comparison_path)

In [ ]:
expected = {
    (dataset_name, model_name, ratio)
    for dataset_name in DATASETS_TO_RUN
    for model_name in MODEL_NAMES
    for ratio in ratio_order
}

completed = {
    (row.Dataset, row.Model, row.SMOTE)
    for row in results_df.itertuples(index=False)
}

missing = sorted(expected - completed)

print("=" * 65)
print("FINAL BENCHMARK VERIFICATION")
print("=" * 65)

print("Expected experiments:", len(expected))
print("Completed experiments:", len(completed))
print("Missing experiments:", len(missing))

if missing:
    print("\nMissing configurations:")
    for item in missing:
        print(item)
else:
    print("\nAll requested experiments completed successfully.")

print("\nOutput folder:")
print(OUTPUT_DIR.resolve())